<a href="https://colab.research.google.com/github/DavidRuiz-beep/Proyecto/blob/main/Taller_GEIH_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**TALLER: IMPORTACIÓN, COMBINACIÓN Y EXPORTACIÓN DE DATOS - GEIH 20**


*   **David Felipe Ruiz Parra**





In [ ]:
# 1. IMPORTAR LIBRERÍAS

import pandas as pd
import numpy as np
import os
import unicodedata
from google.colab import drive

# Mejorar visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


In [ ]:
# 2. CONECTAR CON GOOGLE DRIVE

drive.mount('/content/drive')


In [ ]:
# 3. ESTABLECER RUTA DE LOS DATASETS
ruta_base = '/content/drive/MyDrive/Data_IA/Taller Lim'

print(f"Ruta base: {ruta_base}")
print(f"Contenido de la carpeta: {os.listdir(ruta_base)}")



In [ ]:
# 4. FUNCIONES AUXILIARES

def normalizar_texto(texto):
    """Normaliza texto eliminando tildes y caracteres especiales"""
    if texto is None:
        return ""
    texto = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('ASCII')
    return texto.lower()

def encontrar_archivo_por_mes(mes, ruta_base, tipo):
    """
    Encuentra el archivo correcto para un mes y tipo específico
    tipo: 'carac', 'hogar', 'ftrab', 'ocup'
    """
    ruta_carpeta = f'{ruta_base}/{mes} 2025/CSV'

    # Patrones de búsqueda
    patrones = {
        'carac': ['caracteristicas', 'generales', 'educacion'],
        'hogar': ['datos', 'hogar', 'vivienda'],
        'ftrab': ['fuerza', 'trabajo'],
        'ocup': ['ocupados']
    }

    try:
        archivos = os.listdir(ruta_carpeta)
    except FileNotFoundError:
        return None

    archivos_csv = [a for a in archivos if a.upper().endswith('.CSV')]

    for archivo in archivos_csv:
        archivo_norm = normalizar_texto(archivo)

        if tipo == 'ocup' and 'no' in archivo_norm:
            continue

        if all(normalizar_texto(p) in archivo_norm for p in patrones[tipo]):
            return archivo

    return None


In [ ]:
# 5. FUNCIÓN PARA CARGAR UN MES

def cargar_mes(mes, ruta_base):
    """Carga y une los 4 módulos para un mes específico - Versión con sufijos"""

    print(f"\nCargando {mes}...")
    print("-" * 40)

    # Encontrar archivos
    archivo_carac = encontrar_archivo_por_mes(mes, ruta_base, 'carac')
    archivo_hogar = encontrar_archivo_por_mes(mes, ruta_base, 'hogar')
    archivo_ftrab = encontrar_archivo_por_mes(mes, ruta_base, 'ftrab')
    archivo_ocup = encontrar_archivo_por_mes(mes, ruta_base, 'ocup')

    if None in [archivo_carac, archivo_hogar, archivo_ftrab, archivo_ocup]:
        print(f"Error: No se encontraron todos los archivos para {mes}")
        return None

    ruta_carpeta = f'{ruta_base}/{mes} 2025/CSV'

    # Cargar archivos
    df_carac = pd.read_csv(f'{ruta_carpeta}/{archivo_carac}', encoding='latin-1', sep=';', low_memory=False)
    df_hogar = pd.read_csv(f'{ruta_carpeta}/{archivo_hogar}', encoding='latin-1', sep=';', low_memory=False)
    df_ftrab = pd.read_csv(f'{ruta_carpeta}/{archivo_ftrab}', encoding='latin-1', sep=';', low_memory=False)
    df_ocup = pd.read_csv(f'{ruta_carpeta}/{archivo_ocup}', encoding='latin-1', sep=';', low_memory=False)

    print(f"\nFilas cargadas:")
    print(f"Características: {len(df_carac):,}")
    print(f"Hogar: {len(df_hogar):,}")
    print(f"F. Trabajo: {len(df_ftrab):,}")
    print(f"Ocupados: {len(df_ocup):,}")

    # Unir módulos CON SUFIJOS para evitar columnas duplicadas
    print("\nUniendo módulos...")

    # 1. Características + Hogar (con sufijos)
    df = pd.merge(
        df_carac,
        df_hogar,
        on=['DIRECTORIO', 'SECUENCIA_P', 'HOGAR'],
        how='left',
        suffixes=('', '_hogar'))
    # Eliminar columnas duplicadas de hogar
    cols_hogar = [col for col in df.columns if col.endswith('_hogar')]
    df = df.drop(columns=cols_hogar)
    print(f"Características + Hogar: {len(df):,} filas")

    # 2. + Fuerza Trabajo (con sufijos)
    df = pd.merge(
        df,
        df_ftrab,
        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR'],
        how='left',
        suffixes=('', '_ftrab'))
    cols_ftrab = [col for col in df.columns if col.endswith('_ftrab')]
    df = df.drop(columns=cols_ftrab)
    print(f"+ Fuerza Trabajo: {len(df):,} filas")

    # 3. + Ocupados (con sufijos)
    df = pd.merge(
        df,
        df_ocup,
        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR'],
        how='left',
        suffixes=('', '_ocup'))
    cols_ocup = [col for col in df.columns if col.endswith('_ocup')]
    df = df.drop(columns=cols_ocup)
    print(f"+ Ocupados: {len(df):,} filas")

    # Verificar RAMA4D_R4
    print(f"\n Verificación:")
    print(f"- RAMA4D_R4 presente: {'RAMA4D_R4' in df.columns}")
    if 'RAMA4D_R4' in df.columns:
        print(f"- Ocupados: {df['RAMA4D_R4'].notna().sum():,} ({df['RAMA4D_R4'].notna().sum()/len(df)*100:.1f}%)")

    # Agregar mes
    df['MES'] = mes

    print(f"\nTotal: {len(df):,} filas, {len(df.columns)} columnas")
    print("=" * 50)

    return df


In [ ]:
# 6. CARGAR TODOS LOS MESES

print("="*60)
print("CARGANDO TODOS LOS MESES")
print("="*60)

meses = ['Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
dataframes_meses = []

for mes in meses:
    df_mes = cargar_mes(mes, ruta_base)
    if df_mes is not None:
        dataframes_meses.append(df_mes)

In [ ]:
# 7. CONCATENAR TODOS LOS MESES

if dataframes_meses:
    print("\n" + "="*60)
    print("CONCATENANDO MESES")
    print("="*60)

    geih_2025 = pd.concat(dataframes_meses, ignore_index=True)

    print(f"\nBASE COMPLETA CREADA:")
    print(f"Total filas: {len(geih_2025):,}")
    print(f"Total columnas: {len(geih_2025.columns)}")
    print(f"Meses: {sorted(geih_2025['MES'].unique())}")

    # Distribución por mes
    print("\nDistribución por mes:")
    print(geih_2025['MES'].value_counts().sort_index())

    # Estadísticas de ocupación
    total_ocupados = geih_2025['RAMA4D_R4'].notna().sum()
    print(f"\nOCUPACIÓN GENERAL:")
    print(f"Personas ocupadas: {total_ocupados:,}")
    print(f"Tasa de ocupación: {total_ocupados/len(geih_2025)*100:.1f}%")


In [ ]:
# 8. GUARDAR BASE FINAL
    print("\n" + "="*60)
    print("GUARDANDO BASE FINAL")
    print("="*60)

    archivo_salida = f'{ruta_base}/GEIH_2025_completa.csv'
    geih_2025.to_csv(archivo_salida, index=False, encoding='utf-8')

    print(f"Archivo guardado en: {archivo_salida}")
    print(f"Tamaño: {len(geih_2025):,} filas x {len(geih_2025.columns)} columnas")


In [ ]:
# 9. VERIFICACIÓN RÁPIDA
    print("\n" + "="*60)
    print("VERIFICACIÓN RÁPIDA")
    print("="*60)

    print("\nPrimeras 10 filas:")
    print(geih_2025[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'RAMA4D_R4', 'DPTO', 'MES']].head(10))

    print("\nTALLER COMPLETADO EXITOSAMENTE")

else:
    print("Error: No se pudo cargar ningún mes")